# PyTorch GPU Test + LLM

This notebook contains:
1. **Environment & GPU checks** (auto-detect CUDA / MPS / CPU)  
2. **A minimal neural network** in PyTorch (sanity check + quick training loop)  
3. **LLM example**: auto-download from **ModelScope** and load via **Transformers**  
4. **Generation parameters playground**: greedy, sampling, beam search, reproducibility, and timing

> Tip: Run top-to-bottom. If a package is missing, the notebook will tell you what to install.


## 0) Environment & Utility Helpers

In [ ]:
import os, platform, sys, time
from datetime import datetime

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def timed(fn, *args, **kwargs):
    # Run fn(*args, **kwargs) and return (result, seconds).
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    t1 = time.perf_counter()
    return out, (t1 - t0)

print("Time:", now_str())
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working dir:", os.getcwd())


In [ ]:
# PyTorch device auto-detection: CUDA > MPS (Apple) > CPU
try:
    import torch
except ImportError as e:
    raise SystemExit(
        "PyTorch is not installed.\n"
        "Install (example): pip install torch torchvision torchaudio\n"
        "Then restart the kernel."
    ) from e

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()

print("torch:", torch.__version__)
print("Device:", device)

if device.type == "cuda":
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print("GPU mem (GB):", round(total_gb, 2))
elif device.type == "mps":
    print("MPS is available (Apple Silicon).")
else:
    print("Running on CPU.")


## 1) Minimal Neural Network (PyTorch) — quick sanity check

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

# Tiny synthetic regression dataset: y = Wx + b + noise
N = 2048
in_dim = 16
out_dim = 1

X = torch.randn(N, in_dim)
true_W = torch.randn(in_dim, out_dim)
true_b = torch.randn(out_dim)
y = X @ true_W + true_b + 0.1 * torch.randn(N, out_dim)

X = X.to(device)
y = y.to(device)

class TinyMLP(nn.Module):
    def __init__(self, in_dim, hidden=64, out_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)

model = TinyMLP(in_dim=in_dim, hidden=64, out_dim=out_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Model device:", next(model.parameters()).device)


In [ ]:
# Quick training loop
model.train()
batch_size = 256
steps = 50

def train_steps():
    for step in range(steps):
        idx = torch.randint(0, N, (batch_size,), device=device)
        xb = X[idx]
        yb = y[idx]

        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if (step + 1) % 10 == 0:
            print(f"Step {step+1:03d}/{steps} | loss={loss.item():.6f}")

_, sec = timed(train_steps)
print(f"Training time: {sec:.3f}s on {device}")


In [ ]:
# Quick evaluation
model.eval()
with torch.no_grad():
    pred = model(X[:8])
print("Sample predictions:", pred.squeeze().detach().cpu().tolist())
print("Sample targets:    ", y[:8].squeeze().detach().cpu().tolist())


## 2) LLM Example — Auto-download from ModelScope, then load with Transformers

This section does:

1) `modelscope.snapshot_download()` to download/cache to a local folder  
2) Pass that local folder into `transformers.from_pretrained()`

If packages are missing, install them (example):

```bash
pip install -U modelscope transformers accelerate safetensors
```


In [ ]:
# Import checks
missing = []
try:
    from modelscope import snapshot_download
except Exception:
    missing.append("modelscope")

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
except Exception:
    missing.append("transformers")

if missing:
    raise SystemExit(
        "Missing packages: " + ", ".join(missing) + "\n"
        "Install (example): pip install -U " + " ".join(missing) + " accelerate safetensors\n"
        "Then restart the kernel."
    )

print("Imports OK.")


In [ ]:
import os
import torch
from modelscope import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

model_id = "qwen/Qwen2.5-0.5B"

# Optional: control ModelScope cache location
# os.environ["MODELSCOPE_CACHE"] = os.path.expanduser("~/.cache/modelscope")

print("Download start:", now_str())
model_dir, download_sec = timed(snapshot_download, model_id)
print("Local model_dir:", model_dir)
print(f"Download/cached in: {download_sec:.2f}s")


In [ ]:
# Load tokenizer & model from the local folder
print("Load start:", now_str())

def load_llm():
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_dir,
        device_map="auto",
        torch_dtype="auto",
    )
    return tok, mdl

(tokenizer, llm), load_sec = timed(load_llm)
print(f"Loaded in: {load_sec:.2f}s")

n_params = sum(p.numel() for p in llm.parameters())
first_param = next(llm.parameters())
print("Params:", f"{n_params:,}")
print("dtype:", first_param.dtype, "| device:", first_param.device)

# Avoid pad_token warnings (common for causal LMs)
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Set tokenizer.pad_token = eos_token")


In [ ]:
# Basic generation with timing
prompt = "How should we evaluate the creativity of large language models?"
inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

gen_cfg = GenerationConfig(
    max_new_tokens=120,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.05,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

torch.manual_seed(123)
print("Generate start:", now_str())
with torch.no_grad():
    outputs, infer_sec = timed(llm.generate, **inputs, generation_config=gen_cfg)

print(f"Inference time: {infer_sec:.2f}s")
text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- OUTPUT ---\n")
print(text)


## 3) Generation Parameters Playground (greedy vs sampling vs beam search)

In [ ]:
import time
from transformers import GenerationConfig

def run_gen(label, prompt, seed=0, **kwargs):
    inp = tokenizer(prompt, return_tensors="pt").to(llm.device)
    torch.manual_seed(seed)

    cfg = GenerationConfig(
        max_new_tokens=120,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        **kwargs
    )

    t0 = time.perf_counter()
    with torch.no_grad():
        out = llm.generate(**inp, generation_config=cfg)
    dt = time.perf_counter() - t0

    txt = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"\n=== {label} ===")
    print("Time:", f"{dt:.2f}s")
    # Print the most relevant decoding params
    show = {k: getattr(cfg, k) for k in ["do_sample","temperature","top_p","top_k","num_beams","repetition_penalty","max_new_tokens"] if hasattr(cfg, k)}
    print("Config:", show)
    print(txt)
    return txt, dt

base_prompt = "Explain in 4 bullet points how to evaluate creativity in LLM outputs."
print("Prompt:", base_prompt)


In [ ]:
# 3.1 Greedy decoding (deterministic, typically fastest)
_ = run_gen(
    "Greedy (no sampling)",
    base_prompt,
    do_sample=False,
    num_beams=1,
    repetition_penalty=1.0,
)


In [ ]:
# 3.2 Sampling: temperature + nucleus sampling (top_p)
_ = run_gen(
    "Sampling (temperature=0.8, top_p=0.9)",
    base_prompt,
    seed=123,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    top_k=0,
    repetition_penalty=1.05,
)


In [ ]:
# 3.3 Sampling: temperature + top_k
_ = run_gen(
    "Sampling (temperature=0.8, top_k=40)",
    base_prompt,
    seed=123,
    do_sample=True,
    temperature=0.8,
    top_k=40,
    top_p=1.0,
    repetition_penalty=1.05,
)


In [ ]:
# 3.4 Beam search (often more structured, can be slower)
_ = run_gen(
    "Beam search (num_beams=4)",
    base_prompt,
    do_sample=False,
    num_beams=4,
    repetition_penalty=1.05,
)


## 4) Extra Model / Runtime Info (diagnostics)

In [ ]:
# CUDA memory info (if applicable)
if device.type == "cuda":
    torch.cuda.synchronize()
    allocated = torch.cuda.memory_allocated() / (1024**2)
    reserved = torch.cuda.memory_reserved() / (1024**2)
    print(f"CUDA memory allocated: {allocated:.1f} MB")
    print(f"CUDA memory reserved:   {reserved:.1f} MB")
else:
    print("CUDA memory stats not available. Device:", device)

# Quick config snapshot
print("\nModel config (partial):")
keys = ["model_type", "vocab_size", "hidden_size", "num_hidden_layers", "num_attention_heads", "max_position_embeddings"]
for k in keys:
    if hasattr(llm.config, k):
        print(f"- {k}: {getattr(llm.config, k)}")


## 5) Notes / Tips

- **Download once, reuse forever:** `snapshot_download()` caches locally; next runs should be fast.
- **`device_map="auto"`** uses `accelerate` to place weights across available devices.
- If you want **more deterministic** behavior: use greedy decoding (`do_sample=False`) and keep versions consistent.
- If you see pad token warnings: set `tokenizer.pad_token = tokenizer.eos_token` (already handled above).
